# ResNet-18 on CIFAR-10 — activations in memory

`map()` with no path keeps activations in RAM. Downloads on first run:
CIFAR-10 test split (~20 MB) and ResNet-18 weights (~45 MB).

In [ ]:
import torchvision.transforms as T
from datasets import load_dataset
from torch.utils.data import Dataset
from torchvision.models import ResNet18_Weights, resnet18

from nnact import ActivationMapper, Sample

# ResNet expects 224x224 ImageNet-normalised input; CIFAR ships 32x32.
transform = T.Compose(
    [
        T.Resize(224),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

cifar = load_dataset("uoft-cs/cifar10", split="test")
class_names = cifar.features["label"].names


class CifarSamples(Dataset[Sample]):
    """nnact reads any Dataset yielding Sample(id=..., data=...).

    The id ties a row of activations back to its input, so make it stable.
    """

    def __init__(self, base, n: int) -> None:
        self._base, self._n = base, n

    def __len__(self) -> int:
        return self._n

    def __getitem__(self, idx: int) -> Sample:
        row = self._base[idx]
        return Sample(
            id=f"cifar_{idx:05d}_{class_names[row['label']]}",
            data=transform(row["img"].convert("RGB")),
        )


dataset = CifarSamples(cifar, 512)
print(
    f"{len(dataset)} samples | first id {dataset[0].id!r} | {tuple(dataset[0].data.shape)}"
)

In [ ]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
mapper = ActivationMapper(model)

# Which layers can be hooked. depth=2 descends into the blocks; an unknown
# name raises from map() before the first forward pass, suggesting near misses.
mapper.summary(depth=1)

In [ ]:
# Batches the data, switches to eval, disables grads, shows a progress bar.
store = mapper.map(dataset, ["layer3", "layer4", "avgpool", "fc"], batch_size=64)

store.metadata  # every store records how it was produced

In [ ]:
# Per-sample shape and total bytes per layer. layer3 is ~100x avgpool.
store.summary()

In [ ]:
# A store is a Dataset too: store[i] is one sample across every layer,
# ordered to match layer_names, with position i matching sample_ids[i].
sample = store[0]
print(store.sample_ids[0])
for act in sample.activations:
    print(f"  {act.layer_name:<10} {tuple(act.tensor.shape)}")

# Whole stacked tensor per layer, for anything downstream.
print("\navgpool stacked:", tuple(store.activations["avgpool"].shape))

In [ ]:
# Layer choice dominates storage. One sample is enough to read the shapes
# off summary() and project the cost at any dataset size.
probe = mapper.map(
    CifarSamples(cifar, 1),
    mapper.available_layers(depth=1),
    batch_size=1,
    progress=False,
).summary()

cost = probe[["shape", "elements"]].copy()
cost["KB / sample"] = probe["elements"] * 4 / 1024
cost["GB / 10k imgs"] = probe["elements"] * 4 * 10_000 / 1024**3
cost.round(2)